In [1]:
"""
Pre-Warm Holistic Study — MNIST, Fashion-MNIST, CIFAR-10
─────────────────────────────────────────────────────────
Tests two hypotheses with a single experiment:
  H1: Does Pre-Warm improve final test accuracy?
  H2: Does Pre-Warm reach a given accuracy threshold in fewer steps?

Setup:
  • LR = 3e-3 with cosine annealing → 3e-5 over 5000 steps
  • 5000 steps  (≈21 epochs FMNIST/MNIST, ≈25 epochs CIFAR at bs=256)
  • Eval every 200 steps → 25 checkpoints per run
  • 8 seeds, paired t-tests (one-sided for H1, two-sided for H2 since
    threshold-time can go either way under censoring)
  • Otsu-derived n_patches per dataset; conv1 hyperparams otherwise locked

Drop in after the OG script. Reuses: SimpleCNN, conditioned_init, SEEDS,
device. Reloads data per-dataset.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import tensorflow as tf
import random
from scipy import stats
from skimage.filters import threshold_otsu

from sklearn.cluster import MiniBatchKMeans

# ── Conditioner ───────────────────────────────────────────────────────────────
def extract_patches(images, patch_size, n_patches):
    B, C, H, W = images.shape
    patches = []
    imgs_np = images.cpu().numpy()
    for b in range(B):
        for _ in range(n_patches):
            i = random.randint(0, H - patch_size)
            j = random.randint(0, W - patch_size)
            patch = imgs_np[b, :, i:i+patch_size, j:j+patch_size]
            patches.append(patch.flatten())
    return np.array(patches)


def distance_weights(patch_size):
    center  = patch_size // 2
    weights = []
    for _ in range(1):
        for i in range(patch_size):
            for j in range(patch_size):
                dist = abs(i - center) + abs(j - center)
                weights.append(1.0 / (1.0 + dist))
    return np.array(weights)


def conditioned_init(model, batch_images, n_clusters, n_patches, scale, patch_size, seed):
    patches = extract_patches(batch_images, patch_size, n_patches)
    patches -= patches.mean(axis=1, keepdims=True)

    kmeans    = MiniBatchKMeans(n_clusters=n_clusters, random_state=seed, n_init=3)
    kmeans.fit(patches)
    centroids = kmeans.cluster_centers_.copy()

    dw        = distance_weights(patch_size)
    C_in      = batch_images.shape[1]
    dw_full   = np.tile(dw, C_in)                    # [9] → [27] for CIFAR, unchanged for MNIST
    centroids = centroids * dw_full[np.newaxis, :]
    centroids = centroids / (np.std(centroids) + 1e-8) * scale

    weight_tensor = torch.tensor(
        centroids.reshape(n_clusters, batch_images.shape[1], patch_size, patch_size),
        dtype=torch.float32
    )

    with torch.no_grad():
        target_weight = model.conv1.weight
        out_channels  = target_weight.shape[0]
        if patch_size != 3:
            start         = (patch_size - 3) // 2
            weight_tensor = weight_tensor[:, :, start:start+3, start:start+3].contiguous()
        num_to_copy = min(n_clusters, out_channels)
        target_weight[:num_to_copy].copy_(weight_tensor[:num_to_copy])

    return model

# ── Run config ────────────────────────────────────────────────────────────────
LR_START   = 3e-3
LR_END     = 3e-5
N_STEPS    = 5_000
EVAL_EVERY = 200          # 25 eval checkpoints
BATCH_SIZE = 256
SEEDS = [42, 7, 13, 99, 2025, 1, 8, 21]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Per-dataset normalization, channels, FC dims, accuracy thresholds
DATASET_CFG = {
    'mnist': {
        'loader'    : tf.keras.datasets.mnist.load_data,
        'mean'      : 0.1307, 'std': 0.3081,
        'in_ch'     : 1, 'fc_dim': 64*7*7,
        'thresholds': [0.95, 0.97, 0.98],
    },
    'fashion_mnist': {
        'loader'    : tf.keras.datasets.fashion_mnist.load_data,
        'mean'      : 0.2860, 'std': 0.3530,
        'in_ch'     : 1, 'fc_dim': 64*7*7,
        'thresholds': [0.85, 0.88, 0.90],
    },
    'cifar10': {
        'loader'    : tf.keras.datasets.cifar10.load_data,
        'mean'      : np.array([0.4914, 0.4822, 0.4465]),
        'std'       : np.array([0.2470, 0.2435, 0.2616]),
        'in_ch'     : 3, 'fc_dim': 64*8*8,
        'thresholds': [0.50, 0.60, 0.65],
    },
}

# Locked conv1 hyperparams (n_patches set per dataset by Otsu)
HP_CONV1_BASE = {
    'n_clusters' : 16,
    'scale'      : 0.2,
    'patch_size' : 3,
}


# ── Dataset-flexible CNN ──────────────────────────────────────────────────────
class FlexCNN(nn.Module):
    """Same structure as SimpleCNN but parametrized for input channels and FC dim."""
    def __init__(self, in_ch=1, fc_dim=64*7*7, use_bn1=True):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(fc_dim, 256)
        self.fc2   = nn.Linear(256, 10)
        self.bn1   = nn.BatchNorm2d(32) if use_bn1 else nn.Identity()
        self.bn2   = nn.BatchNorm2d(64)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


# ── Dataset preparation ───────────────────────────────────────────────────────
def prepare_dataset(name):
    cfg = DATASET_CFG[name]
    (xtr, ytr), (xte, yte) = cfg['loader']()

    # Squeeze label arrays (CIFAR returns shape (N,1))
    ytr = np.array(ytr).squeeze().astype(np.int64)
    yte = np.array(yte).squeeze().astype(np.int64)

    # Raw [0,1] for Otsu, then normalize for training
    xtr_raw = xtr.astype(np.float32) / 255.0
    xte_raw = xte.astype(np.float32) / 255.0

    # Otsu foreground density on raw pixels
    # For multi-channel, average over channels per-pixel before Otsu
    if cfg['in_ch'] == 1:
        flat = xtr_raw.flatten()
    else:
        flat = xtr_raw.mean(axis=-1).flatten()  # average RGB → luminance proxy
    tau = threshold_otsu(flat)
    density = float((flat > tau).mean())
    n_patches = max(1, int(np.floor((32 / 4) * (1.0 / density))))  # F=32 conv1 filters

    # Normalize
    mean, std = cfg['mean'], cfg['std']
    if cfg['in_ch'] == 1:
        xtr = (xtr_raw - mean) / std
        xte = (xte_raw - mean) / std
        xtr = xtr[:, np.newaxis, :, :]
        xte = xte[:, np.newaxis, :, :]
    else:
        # Broadcast per-channel stats over (H, W, C), then transpose to (C, H, W)
        xtr = (xtr_raw - mean) / std
        xte = (xte_raw - mean) / std
        xtr = xtr.transpose(0, 3, 1, 2)
        xte = xte.transpose(0, 3, 1, 2)

    train_ds = torch.utils.data.TensorDataset(torch.tensor(xtr.astype(np.float32)),
                                              torch.tensor(ytr))
    test_ds  = torch.utils.data.TensorDataset(torch.tensor(xte.astype(np.float32)),
                                              torch.tensor(yte))
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE,
                                               shuffle=True,  num_workers=2)
    test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=512,
                                               shuffle=False, num_workers=2)

    return {
        'name'        : name,
        'train_loader': train_loader,
        'test_loader' : test_loader,
        'in_ch'       : cfg['in_ch'],
        'fc_dim'      : cfg['fc_dim'],
        'density'     : density,
        'otsu_tau'    : float(tau),
        'n_patches'   : n_patches,
        'thresholds'  : cfg['thresholds'],
    }


# ── Evaluation ────────────────────────────────────────────────────────────────
def evaluate(model, test_loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return correct / total


# ── Single run with periodic eval ────────────────────────────────────────────
def run_with_curve(seed, ds, conditioned, hp):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)

    model     = FlexCNN(in_ch=ds['in_ch'], fc_dim=ds['fc_dim']).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_START)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=N_STEPS, eta_min=LR_END
    )

    # Grab first batch for conditioning
    first_iter   = iter(ds['train_loader'])
    images0, _   = next(first_iter)
    if conditioned and hp is not None:
        model = conditioned_init(model, images0, **hp, seed=seed)

    losses     = []
    eval_steps = []
    eval_accs  = []
    step = 0
    model.train()
    while step < N_STEPS:
        for images, labels in ds['train_loader']:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            losses.append(loss.item())
            step += 1

            if step % EVAL_EVERY == 0 or step == N_STEPS:
                acc = evaluate(model, ds['test_loader'])
                eval_steps.append(step)
                eval_accs.append(acc)
                model.train()

            if step >= N_STEPS:
                break

    return {
        'losses'     : np.array(losses),
        'eval_steps' : np.array(eval_steps),
        'eval_accs'  : np.array(eval_accs),
        'final_loss' : losses[-1],
        'final_acc'  : eval_accs[-1],
    }


# ── Time-to-threshold helper ─────────────────────────────────────────────────
def steps_to_threshold(eval_steps, eval_accs, threshold):
    """First step where acc >= threshold. Returns N_STEPS+1 if never reached (right-censored)."""
    hits = np.where(eval_accs >= threshold)[0]
    if len(hits) == 0:
        return N_STEPS + 1
    return int(eval_steps[hits[0]])


# ── Per-dataset experiment ───────────────────────────────────────────────────
def run_dataset(ds_name):
    print('\n' + '█'*78)
    print(f'  DATASET: {ds_name}')
    print('█'*78)

    ds = prepare_dataset(ds_name)
    print(f'  Otsu threshold τ* = {ds["otsu_tau"]:.4f}')
    print(f'  Foreground density d = {ds["density"]:.4f}')
    print(f'  n_patches (formula) = ⌊(32/4)·(1/{ds["density"]:.4f})⌋ = {ds["n_patches"]}')

    hp = {**HP_CONV1_BASE, 'n_patches': ds['n_patches']}
    print(f'  Conv1 HP: {hp}')

    base_runs, cond_runs = [], []
    for i, s in enumerate(SEEDS):
        print(f'  Seed {s} ({i+1}/{len(SEEDS)})... ', end='', flush=True)
        br = run_with_curve(s, ds, conditioned=False, hp=None)
        cr = run_with_curve(s, ds, conditioned=True,  hp=hp)
        base_runs.append(br); cond_runs.append(cr)
        print(f'base_acc={br["final_acc"]:.4f}  cond_acc={cr["final_acc"]:.4f}  '
              f'Δ={cr["final_acc"]-br["final_acc"]:+.4f}')

    return ds, base_runs, cond_runs


# ── Analysis: H1 (final acc) + H2 (time-to-threshold) ────────────────────────
def analyze(ds, base_runs, cond_runs):
    name = ds['name']
    print('\n' + '─'*78)
    print(f'  ANALYSIS — {name}')
    print('─'*78)

    base_final_acc  = np.array([r['final_acc']  for r in base_runs])
    cond_final_acc  = np.array([r['final_acc']  for r in cond_runs])
    base_final_loss = np.array([r['final_loss'] for r in base_runs])
    cond_final_loss = np.array([r['final_loss'] for r in cond_runs])

    # ── H1: final accuracy ──
    delta_acc       = cond_final_acc.mean() - base_final_acc.mean()
    _, p_acc        = stats.ttest_rel(cond_final_acc, base_final_acc, alternative='greater')
    delta_loss      = base_final_loss.mean() - cond_final_loss.mean()
    _, p_loss       = stats.ttest_rel(base_final_loss, cond_final_loss, alternative='greater')
    n_wins_acc      = int((cond_final_acc > base_final_acc).sum())

    print(f'\nH1 — Final metrics at step {N_STEPS}:')
    print(f'  Kaiming  : loss={base_final_loss.mean():.4f}  acc={base_final_acc.mean():.4f} ± {base_final_acc.std():.4f}')
    print(f'  Pre-Warm : loss={cond_final_loss.mean():.4f}  acc={cond_final_acc.mean():.4f} ± {cond_final_acc.std():.4f}')
    print(f'  Δacc  = {delta_acc:+.4f}   p={p_acc:.4f}   ({n_wins_acc}/{len(SEEDS)} wins)')
    print(f'  Δloss = {delta_loss:+.4f}   p={p_loss:.4f}')

    # ── H2: time-to-threshold ──
    print(f'\nH2 — Steps to reach accuracy threshold (paired):')
    print(f'  {"thresh":>7}  {"base_avg":>9}  {"cond_avg":>9}  {"Δsteps":>8}  '
          f'{"p":>7}  {"base_reach":>11}  {"cond_reach":>11}')
    h2_results = []
    for thr in ds['thresholds']:
        base_steps = np.array([steps_to_threshold(r['eval_steps'], r['eval_accs'], thr)
                               for r in base_runs])
        cond_steps = np.array([steps_to_threshold(r['eval_steps'], r['eval_accs'], thr)
                               for r in cond_runs])
        base_reach = int((base_steps <= N_STEPS).sum())
        cond_reach = int((cond_steps <= N_STEPS).sum())

        # Only test pairs where BOTH methods reached the threshold (avoids censoring bias)
        both_reached = (base_steps <= N_STEPS) & (cond_steps <= N_STEPS)
        if both_reached.sum() >= 3:
            # One-sided: cond_steps < base_steps means Pre-Warm reached faster
            _, p_steps = stats.ttest_rel(base_steps[both_reached], cond_steps[both_reached],
                                          alternative='greater')
            delta_steps_mean = base_steps[both_reached].mean() - cond_steps[both_reached].mean()
        else:
            p_steps = float('nan')
            delta_steps_mean = float('nan')

        h2_results.append({'thresh': thr, 'base_steps': base_steps, 'cond_steps': cond_steps,
                           'delta': delta_steps_mean, 'p': p_steps,
                           'base_reach': base_reach, 'cond_reach': cond_reach})
        delta_str = f'{delta_steps_mean:+8.1f}' if not np.isnan(delta_steps_mean) else '     n/a'
        p_str     = f'{p_steps:.4f}'             if not np.isnan(p_steps)         else '   n/a '
        print(f'  {thr:>7.2f}  '
              f'{base_steps[base_steps<=N_STEPS].mean() if base_reach else float("nan"):>9.1f}  '
              f'{cond_steps[cond_steps<=N_STEPS].mean() if cond_reach else float("nan"):>9.1f}  '
              f'{delta_str}  {p_str}  '
              f'{base_reach:>5}/{len(SEEDS)}    {cond_reach:>5}/{len(SEEDS)}')

    return {
        'name': name,
        'H1' : {'delta_acc': delta_acc, 'p_acc': p_acc,
                'delta_loss': delta_loss, 'p_loss': p_loss,
                'n_wins': n_wins_acc,
                'base_acc': base_final_acc, 'cond_acc': cond_final_acc},
        'H2' : h2_results,
        'curves': {
            'eval_steps' : base_runs[0]['eval_steps'],
            'base_accs'  : np.array([r['eval_accs'] for r in base_runs]),
            'cond_accs'  : np.array([r['eval_accs'] for r in cond_runs]),
        }
    }


# # ── Main ─────────────────────────────────────────────────────────────────────
# print('='*78)
# print('  Pre-Warm Holistic Study')
# print(f'  LR: cosine {LR_START} → {LR_END} over {N_STEPS} steps')
# print(f'  Eval every {EVAL_EVERY} steps, {len(SEEDS)} seeds')
# print('='*78)

# all_results = {}
# for ds_name in ['mnist', 'fashion_mnist', 'cifar10']:
#     ds, base_runs, cond_runs = run_dataset(ds_name)
#     all_results[ds_name] = analyze(ds, base_runs, cond_runs)

# # ── Cross-dataset summary ────────────────────────────────────────────────────
# print('\n\n' + '═'*78)
# print('  CROSS-DATASET SUMMARY')
# print('═'*78)

# print('\nH1 — Final accuracy:')
# print(f'  {"dataset":<16}  {"base_acc":>9}  {"cond_acc":>9}  {"Δacc":>8}  {"p_acc":>7}  wins')
# for ds_name, R in all_results.items():
#     print(f'  {ds_name:<16}  {R["H1"]["base_acc"].mean():>9.4f}  '
#           f'{R["H1"]["cond_acc"].mean():>9.4f}  {R["H1"]["delta_acc"]:>+8.4f}  '
#           f'{R["H1"]["p_acc"]:>7.4f}  {R["H1"]["n_wins"]}/{len(SEEDS)}')

# print('\nH2 — Time-to-threshold (steps saved by Pre-Warm; positive = PW reached faster):')
# print(f'  {"dataset":<16}  {"threshold":>9}  {"Δsteps":>9}  {"p":>7}  ...')
# for ds_name, R in all_results.items():
#     for h2 in R['H2']:
#         delta_str = f'{h2["delta"]:>+9.1f}' if not np.isnan(h2['delta']) else '      n/a'
#         p_str     = f'{h2["p"]:>7.4f}'      if not np.isnan(h2['p'])     else '    n/a'
#         print(f'  {ds_name:<16}  {h2["thresh"]:>9.2f}  {delta_str}  {p_str}  '
#               f'{h2["base_reach"]:>5}/{len(SEEDS)}    {h2["cond_reach"]:>5}/{len(SEEDS)}')

# print('\nLegend:')
# print('  H1 p_acc: one-sided paired t-test, alternative = Pre-Warm > Kaiming')
# print('  H2 p:     one-sided paired t-test on seeds where BOTH methods reached threshold')
# print('  Δsteps positive → Pre-Warm needed fewer steps to reach threshold')
# print('\nDone.')

Device: cuda


Device: cuda
==============================================================================
  Pre-Warm Holistic Study
  LR: cosine 0.003 → 3e-05 over 5000 steps
  Eval every 200 steps, 8 seeds
==============================================================================

██████████████████████████████████████████████████████████████████████████████
  DATASET: mnist
██████████████████████████████████████████████████████████████████████████████
  Otsu threshold τ* = 0.4434
  Foreground density d = 0.1373
  n_patches (formula) = ⌊(32/4)·(1/0.1373)⌋ = 58
  Conv1 HP: {'n_clusters': 16, 'scale': 0.2, 'patch_size': 3, 'n_patches': 58}
  Seed 42 (1/8)... base_acc=0.9930  cond_acc=0.9938  Δ=+0.0008
  Seed 7 (2/8)... base_acc=0.9927  cond_acc=0.9935  Δ=+0.0008
  Seed 13 (3/8)... base_acc=0.9926  cond_acc=0.9931  Δ=+0.0005
  Seed 99 (4/8)... base_acc=0.9929  cond_acc=0.9936  Δ=+0.0007
  Seed 2025 (5/8)... base_acc=0.9927  cond_acc=0.9932  Δ=+0.0005
  Seed 1 (6/8)... base_acc=0.9926  cond_acc=0.9935  Δ=+0.0009
  Seed 8 (7/8)... base_acc=0.9920  cond_acc=0.9929  Δ=+0.0009
  Seed 21 (8/8)... base_acc=0.9936  cond_acc=0.9929  Δ=-0.0007

──────────────────────────────────────────────────────────────────────────────
  ANALYSIS — mnist
──────────────────────────────────────────────────────────────────────────────

H1 — Final metrics at step 5000:
  Kaiming  : loss=0.0001  acc=0.9928 ± 0.0004
  Pre-Warm : loss=0.0001  acc=0.9933 ± 0.0003
  Δacc  = +0.0006   p=0.0109   (7/8 wins)
  Δloss = -0.0000   p=0.9843

H2 — Steps to reach accuracy threshold (paired):
   thresh   base_avg   cond_avg    Δsteps        p   base_reach   cond_reach
     0.95      200.0      200.0      +0.0     n/a       8/8        8/8
     0.97      200.0      200.0      +0.0     n/a       8/8        8/8
     0.98      275.0      250.0     +25.0  0.2992      8/8        8/8

██████████████████████████████████████████████████████████████████████████████
  DATASET: fashion_mnist
██████████████████████████████████████████████████████████████████████████████
Downloading data from https://storage.googleapis.com/tensorflow/tf-keras-datasets/train-labels-idx1-ubyte.gz
29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Downloading data from https://storage.googleapis.com/tensorflow/tf-keras-datasets/train-images-idx3-ubyte.gz
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Downloading data from https://storage.googleapis.com/tensorflow/tf-keras-datasets/t10k-labels-idx1-ubyte.gz
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Downloading data from https://storage.googleapis.com/tensorflow/tf-keras-datasets/t10k-images-idx3-ubyte.gz
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
  Otsu threshold τ* = 0.3809
  Foreground density d = 0.3602
  n_patches (formula) = ⌊(32/4)·(1/0.3602)⌋ = 22
  Conv1 HP: {'n_clusters': 16, 'scale': 0.2, 'patch_size': 3, 'n_patches': 22}
  Seed 42 (1/8)... base_acc=0.9278  cond_acc=0.9295  Δ=+0.0017
  Seed 7 (2/8)... base_acc=0.9271  cond_acc=0.9305  Δ=+0.0034
  Seed 13 (3/8)... base_acc=0.9276  cond_acc=0.9293  Δ=+0.0017
  Seed 99 (4/8)... base_acc=0.9251  cond_acc=0.9276  Δ=+0.0025
  Seed 2025 (5/8)... base_acc=0.9292  cond_acc=0.9284  Δ=-0.0008
  Seed 1 (6/8)... base_acc=0.9282  cond_acc=0.9266  Δ=-0.0016
  Seed 8 (7/8)... base_acc=0.9255  cond_acc=0.9268  Δ=+0.0013
  Seed 21 (8/8)... base_acc=0.9231  cond_acc=0.9276  Δ=+0.0045

──────────────────────────────────────────────────────────────────────────────
  ANALYSIS — fashion_mnist
──────────────────────────────────────────────────────────────────────────────

H1 — Final metrics at step 5000:
  Kaiming  : loss=0.0087  acc=0.9267 ± 0.0019
  Pre-Warm : loss=0.0123  acc=0.9283 ± 0.0013
  Δacc  = +0.0016   p=0.0308   (6/8 wins)
  Δloss = -0.0036   p=0.8740

H2 — Steps to reach accuracy threshold (paired):
   thresh   base_avg   cond_avg    Δsteps        p   base_reach   cond_reach
     0.85      200.0      200.0      +0.0     n/a       8/8        8/8
     0.88      350.0      375.0     -25.0  0.8247      8/8        8/8
     0.90      625.0      625.0      +0.0  0.5000      8/8        8/8

██████████████████████████████████████████████████████████████████████████████
  DATASET: cifar10
██████████████████████████████████████████████████████████████████████████████
Downloading data from https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz
  1236992/170498071 ━━━━━━━━━━━━━━━━━━━━ 1:15:47 27us/step